# 📡 Conocimiento estático y búsqueda web

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 1 — 35 minutos**

---

## Objetivo

Entender por qué los LLMs "no saben" cosas recientes — y ver la primera estrategia para solucionarlo.

## Al terminar este bloque vas a poder:

1. Explicar el concepto de *cutoff date* y sus consecuencias prácticas.
2. Conectar una API de búsqueda web para inyectar información actual en un prompt.
3. Construir un pipeline sencillo: pregunta → búsqueda → respuesta fundamentada.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Conocimiento estático** | Todo lo que el LLM aprendió durante entrenamiento — congelado en el tiempo. |
| **Cutoff date** | La fecha hasta la cual el modelo vio datos. Después de ahí, "no sabe" nada nuevo. |
| **Alucinación** | Cuando el modelo genera información plausible pero falsa o desactualizada con total confianza. |
| **Web Search Integration** | Patrón que consulta la web antes del LLM para darle contexto fresco. |
| **RAG** | Retrieval-Augmented Generation — el nombre formal del patrón que une recuperación + generación. |

## El problema: los LLMs tienen memoria congelada

### Analogía

Un LLM es como un experto que estudió muchísimo hasta cierta fecha, se encerró en un cuarto sin internet, y desde ahí responde preguntas. Todo lo que pasó después de que cerró la puerta, simplemente no lo sabe — y si le preguntás, va a inventar algo que suene razonable.

### Dónde vive esto en el mundo real

- GPT-4 tiene un cutoff de principios de 2024. Si le preguntás quién ganó las elecciones de 2025, va a "alucinar".
- Perplexity, Bing Chat y ChatGPT con búsqueda web resuelven esto conectando un motor de búsqueda antes de responder.
- En sistemas RAG empresariales (lo que vamos a construir hoy), el "buscador" no es Google sino tu propia base de documentos.

### El patrón de solución

```
[Pregunta del usuario]
        ↓
[Búsqueda web / base de documentos]
        ↓
[Contexto actualizado → LLM]
        ↓
[Respuesta fundamentada]
```

Hoy vamos a ver la versión con búsqueda web. En los próximos bloques, el buscador va a ser ChromaDB con tus propios documentos.

### ✎ Para pensar

- ¿Qué diferencia hay entre que el modelo "no sepa" algo y que lo "invente"? ¿Cuál es más peligroso?
- Si el modelo tiene acceso a búsqueda web, ¿podría buscar información errónea y tomarla como verdadera?

## Configuración

Necesitás dos API keys:
- **SERP API**: busca resultados de Google de forma programática
- **OpenAI**: genera la respuesta final usando GPT

Las keys se guardan en los secretos de Colab (ícono de llave 🔑 en la barra izquierda). **Nunca las pongas directamente en el código ni las subas a git.**

In [ ]:
# Instalamos las librerías
!pip install google-search-results openai

In [ ]:
# Paso 1. Registrarse a SERP API y Open AI para obetener las siguientes claves

from google.colab import userdata

SERP_API_KEY = userdata.get("SERP_API_KEY")
#OPENAI_ORGANIZATION = userdata.get("OPENAI_ORGANIZATION")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

## Paso 1 — Buscar en la web

La función `serp_results()` hace una búsqueda en Google a través de SERP API y devuelve los snippets de los primeros resultados. Cada resultado tiene URL, título y un fragmento de texto.

In [ ]:
from itertools import islice
from dataclasses import dataclass
from serpapi import GoogleSearch

@dataclass(frozen=True)
class SnippetCitation:
    Url: str
    Title: str
    Snippet: str

def serp_results(query: str, num=5, api_key=SERP_API_KEY):
  params = {
      "engine": "google",
      "q": query,
      "api_key": api_key,
      "num":num
  }
  search = GoogleSearch(params)
  res = search.get_dict()
  organic_results = res.get("organic_results", []) # Add this line to safely access the key
  return_results = []
  for result in organic_results:
      try:
        return_results.append(SnippetCitation(Url=result["link"], Title=result["title"], Snippet=result["snippet"]))
      except Exception as e:
          print(e)
  return return_results

In [ ]:
serp_res = serp_results("¿Cual es el nombre del Papa?")
serp_res

## Paso 2 — Construir el prompt con contexto

El truco es simple: tomamos los snippets de búsqueda y los pegamos en el prompt como "contexto". El modelo tiene instrucciones de responder **solo** con lo que está en ese contexto — y de decir que no sabe si la respuesta no está.

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
  # Uses OPENAI_API_KEY environment variable

def chat_complete(
    syst: str | None,
    user: list[str] = [],
    assistant: list[str] = [],
    max_tokens: int = 1024,
    temperature: float = 0,
    model: str = "gpt-4o",
) -> str:
    # Initialize the OpenAI client
    messages: list[dict[str, str]] = []

    if syst is not None:
        messages.append({"role": "system", "content": syst})

    for i in range(len(user)):
        messages.append({"role": "user", "content": user[i]})
        if len(assistant) > i:
            messages.append({"role": "assistant", "content": assistant[i]})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )

    return response.choices[0].message.content

In [ ]:

SYSTEM_PROMPT = """Tu objetivo es responder a la pregunta [PREGUNTA] utilizando este contexto [CONTEXTO].

Si la respuesta no se encuentra en el contexto, responde que no sabes. """


In [ ]:
pregunta = "¿Cual es el nombre del Papa?"

In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT.replace("[PREGUNTA]", pregunta)
SYSTEM_PROMPT = SYSTEM_PROMPT.replace("[CONTEXTO]", str(serp_res))

In [ ]:
chat_complete("Hola")

### ✎ Para pensar

- ¿Por qué es importante la instrucción "si la respuesta no se encuentra en el contexto, decí que no sabés"?
- ¿Qué pasaría si el modelo pudiera responder tanto con su conocimiento interno como con el contexto?

## Paso 3 — Salida estructurada

Hasta ahora el modelo devuelve texto libre. En aplicaciones reales necesitamos que devuelva **JSON válido** para que nuestro código lo procese. Esto se logra con un schema que define exactamente qué campos esperar.

In [ ]:
chat_complete(SYSTEM_PROMPT)

In [ ]:
def limpiar_markdown(content: str):
    # Remove markdown code block markers
    content = content.strip()
    if content.startswith('```json'):
        content = content[7:]  # Remove ```json
    elif content.startswith('```'):
        content = content[3:]   # Remove ```
    if content.endswith('```'):
        content = content[:-3]  # Remove closing ```
    content = content.strip()
    return content

In [ ]:
def chat_complete(
    syst: str | None,
    user: list[str] = [],
    assistant: list[str] = [],
    max_tokens: int = 1024,
    temperature: float = 0,
    model: str = "gpt-4o",
    schema: dict | None = None
) -> str:
    # Initialize the OpenAI client
    messages: list[dict[str, str]] = []

    if syst is not None:
        messages.append({"role": "system", "content": syst})

    for i in range(len(user)):
        messages.append({"role": "user", "content": user[i]})
        if len(assistant) > i:
            messages.append({"role": "assistant", "content": assistant[i]})

    # Build the request parameters
    request_params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if schema is not None:
        request_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "structured_response",
                "schema": schema,
                "strict": True
            }
        }
    response = client.chat.completions.create(**request_params)
    content = response.choices[0].message.content
    content = limpiar_markdown(content)
    return content


In [ ]:
SYSTEM_PROMPT = """Tu objetivo es responder a la pregunta [PREGUNTA] utilizando este contexto [CONTEXTO].

El formato de salida debe ser un array de json con todas las entidades mencionadas. Cada una representada como un único string del título """

In [ ]:
pregunta = "Recomendaciones de series 2025"

In [ ]:
# Ahora representemos en un diccionario un esquema con un array de string
RECOMMENDATIONS_SCHEMA = {
    "type": "ARRAY",
    "items": {
        "type": "STRING"
    }
}

### ✎ Para pensar

- Si el modelo a veces genera texto antes o después del JSON, ¿qué implica eso para un sistema de producción?
- ¿En qué casos preferirías texto libre y en cuáles JSON estructurado?

## ⛰️ Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **Cutoff date** | El modelo no sabe nada posterior a su fecha de entrenamiento |
| **Alucinación** | El modelo genera respuestas plausibles pero potencialmente falsas |
| **Web search** | Inyectar snippets de búsqueda en el prompt como contexto |
| **Prompt con contexto** | El patrón: `[contexto] + [pregunta]` → respuesta fundamentada |
| **JSON estructurado** | Forzar salida en schema para procesar programáticamente |

**Próximo bloque →** Ollama: vamos a correr LLMs localmente en tu computadora — sin API key, sin costo, sin internet.